# Notebook-first traffic forecasting candidate workflow

Status: executable workflow source, **not evidence of a completed training run**. No dataset is bundled. Install the project with its notebook/forecast dependencies and select a lawfully acquired, audited feature snapshot. Run from a clean kernel in order. Training and candidate evaluation execute here; shared package helpers also serve inference. No Airflow training or automatic champion update.

One run uses one dataset namespace. Vietnamese and METR-LA results remain separate; a benchmark model is never the Vietnamese champion without local evaluation. Prepared speed features/targets must already use km/h and preserve original source-unit conversion provenance. +15/+60 are MVP horizons. See [requirements](../docs/requirements/cognitive-traffic-mvp-v1.md).


## 1. Declare run inputs, split, seed and provenance before inspecting results

Set explicit paths/times using your audited dataset. `.local/` is the local output root; Kaggle uses `/kaggle/working`. Do not commit data, artifacts or credentials. Keep test boundaries fixed before model selection. The final seven days are held out; maximum horizon is purged before that boundary.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import subprocess
from datetime import UTC, datetime

import joblib
import numpy as np
import pandas as pd

from cognitive_traffic.forecasting.baselines import HistoricalMedian, Persistence, SeasonalMedian
from cognitive_traffic.forecasting.evaluation import WalkForwardSplitter, pinball_loss, regression_metrics
from cognitive_traffic.forecasting.training import train_quantile_bundle

DATASET_ID = "vn-mvp-12"  # Or metr-la, in a separate run/output directory.
DATASET_PATH = Path(".local/datasets/feature_snapshot.parquet")
SOURCE_MANIFEST_PATH = Path(".local/datasets/manifest.json")
HORIZONS = (15, 60)
SEED = 42
FEATURE_VERSION = "1"
SCHEMA_VERSION = "1"
HOLDOUT_DAYS = 7
MINIMUM_TRAIN_DAYS = 28
VALIDATION_DAYS = 7
# Explicit audit reference; do not replace with an unsupported passing flag.
SOURCE_AUDIT_REFERENCE = ""  # Rights, mapping, units, availability/target construction.
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/input").exists() else Path(".local/models")
RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = OUTPUT_ROOT / DATASET_ID / RUN_ID
assert DATASET_ID in {"vn-mvp-12", "metr-la"}
assert SOURCE_AUDIT_REFERENCE, "Provide the dataset audit evidence before training"
np.random.seed(SEED)


## 2. Load and audit the prepared dataset

No fallback CSV, synthetic observations or inferred target columns. The source manifest must identify the namespace and km/h units; its objects/split/source window/rights and feature-input availability require independent review. Rows are unique by road and prediction time. The check below checks declared availability only; it cannot establish truthful provider timestamps or upstream point-in-time feature construction.

In [ ]:
source_manifest = json.loads(SOURCE_MANIFEST_PATH.read_text())
assert source_manifest["dataset_id"] == DATASET_ID
assert source_manifest["unit"] == "km/h", "Convert verified mph inputs explicitly during preparation"
df = pd.read_parquet(DATASET_PATH).reset_index(drop=True)
targets = {h: f"target_speed_{h}m" for h in HORIZONS}
required = {"road_id", "prediction_time", "available_at", "speed_kmh", *targets.values()}
assert required <= set(df), f"Missing columns: {required - set(df)}"
assert not df.empty
for column in ("prediction_time", "available_at"):
    df[column] = pd.to_datetime(df[column], utc=True, errors="raise")
assert not df.duplicated(["road_id", "prediction_time"]).any()
assert (df["available_at"] <= df["prediction_time"]).all()
assert (df[list(targets.values())].dropna() >= 0).all().all()
df = df.sort_values(["prediction_time", "road_id"]).reset_index(drop=True)
display(df.head())
display(df.isna().mean().sort_values(ascending=False).to_frame("missing_ratio"))
print({"rows": len(df), "roads": df.road_id.nunique(), "dataset_id": DATASET_ID})


## 3. Freeze temporal split and train using shared helpers

The final holdout is never passed to training. Purge the largest target horizon before holdout. Package walk-forward scores remain development diagnostics: the current splitter does not purge every internal fold horizon, so they are not leakage-certified selection evidence. The independent final holdout is the candidate report. Threshold selection must use development data, never the held-out result.

In [ ]:
holdout_start = df.prediction_time.max() - pd.Timedelta(days=HOLDOUT_DAYS)
train_cutoff = holdout_start - pd.Timedelta(minutes=max(HORIZONS))
train = df[df.prediction_time < train_cutoff].copy()
holdout = df[df.prediction_time >= holdout_start].copy()
assert not train.empty and not holdout.empty
splitter = WalkForwardSplitter(minimum_train_days=MINIMUM_TRAIN_DAYS,
                              validation_days=VALIDATION_DAYS, step_days=7)
folds = splitter.split(train)
split_manifest = {
    "holdout_start": holdout_start.isoformat(), "train_cutoff": train_cutoff.isoformat(),
    "purge_minutes": max(HORIZONS), "train_rows": len(train), "holdout_rows": len(holdout),
    "folds": [{"number": f.number, "train_start": f.train_start.isoformat(),
               "train_end": f.train_end.isoformat(), "validation_start": f.validation_start.isoformat(),
               "validation_end": f.validation_end.isoformat()} for f in folds],
}
bundle = train_quantile_bundle(train, targets=targets, splitter=splitter,
                               feature_version=FEATURE_VERSION, schema_version=SCHEMA_VERSION,
                               random_state=SEED)


## 4. Evaluate common holdout cohort, baselines, intervals and missingness

Report sample counts per horizon and coverage loss. P10–p90 empirical coverage/width and pinball loss are held-out measurements, not guaranteed calibration or congestion probability. Peak hours below use Vietnam time explicitly. A benchmark requires its own documented local peak-hour policy; the default must not become a Vietnamese promotion gate.

In [ ]:
rows = []
prediction_frames = []
for horizon, target in targets.items():
    cohort = holdout.dropna(subset=[target, "speed_kmh", *bundle.feature_columns]).copy()
    assert not cohort.empty, f"No eligible holdout rows for {horizon}m"
    quantiles = bundle.predict(cohort, horizon)
    actual = cohort[target]
    for baseline in (Persistence(), HistoricalMedian(), SeasonalMedian()):
        fit_frame = train.dropna(subset=[target, "speed_kmh"])
        baseline.fit(fit_frame, target)
        rows.append({"horizon_minutes": horizon, "model": baseline.name,
                     "count": len(cohort), **regression_metrics(actual, baseline.predict(cohort))})
    rows.append({"horizon_minutes": horizon, "model": "lightgbm_p50",
                 "count": len(cohort), "eligible_fraction": len(cohort) / len(holdout),
                 **regression_metrics(actual, quantiles.p50),
                 "interval_coverage": float(((actual >= quantiles.p10) & (actual <= quantiles.p90)).mean()),
                 "interval_width": float((quantiles.p90 - quantiles.p10).mean()),
                 **{f"pinball_p{q}": pinball_loss(actual, quantiles[f"p{q}"], q / 100)
                    for q in (10, 50, 90)}})
    predicted = cohort[["road_id", "prediction_time"]].copy()
    predicted["horizon_minutes"] = horizon
    predicted["actual"] = actual
    predicted[["p10", "p50", "p90"]] = quantiles
    prediction_frames.append(predicted)
metrics = pd.DataFrame(rows)
predictions = pd.concat(prediction_frames, ignore_index=True)
local_hour = predictions.prediction_time.dt.tz_convert("Asia/Ho_Chi_Minh").dt.hour
peak = predictions[local_hour.isin([7, 8, 9, 16, 17, 18])]
peak_mae = regression_metrics(peak.actual, peak.p50)["mae"] if not peak.empty else None
display(metrics)
print({"peak_mae_vietnam_policy": peak_mae, "peak_samples": len(peak)})
for dimension in ("road_id", "horizon_minutes"):
    display(predictions.assign(absolute_error=(predictions.p50 - predictions.actual).abs())
            .groupby(dimension).absolute_error.agg(["count", "mean", "median"]))


## 5. Export independent candidate, checksums and environment

Bundle and per-horizon metrics are independent of the notebook kernel. Development fold metrics are labelled separately from the held-out report. `candidate.json` is review input only. Upload the exact bytes and preserve declared object keys if using object storage. Provide separate measured leakage/DQ/serving-signature reports bound to its SHA-256 for Airflow review; this notebook does not manufacture passed gates or update any champion alias.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
artifact_path = OUTPUT_DIR / "bundle.joblib"
metrics_path = OUTPUT_DIR / "metrics.json"
joblib.dump(bundle, artifact_path, compress=3)
metrics_path.write_text(metrics.to_json(orient="records", indent=2))
predictions.to_parquet(OUTPUT_DIR / "holdout_predictions.parquet", index=False)
(OUTPUT_DIR / "development_metrics.json").write_text(json.dumps(bundle.metrics, indent=2))
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
try:
    code_revision = subprocess.run(["git", "rev-parse", "HEAD"], check=True,
                                   capture_output=True, text=True).stdout.strip()
    working_tree_dirty = bool(subprocess.run(["git", "status", "--porcelain"], check=True,
                                            capture_output=True, text=True).stdout.strip())
except (OSError, subprocess.CalledProcessError):
    code_revision, working_tree_dirty = "unavailable", None
candidate = {
    "execution_origin": "notebook", "dataset_id": DATASET_ID, "run_id": RUN_ID,
    "horizons_minutes": list(HORIZONS), "unit": "km/h", "seed": SEED,
    "feature_columns": list(bundle.feature_columns), "feature_version": FEATURE_VERSION,
    "schema_version": SCHEMA_VERSION, "split": split_manifest,
    "source_manifest": source_manifest, "source_manifest_sha256": sha256(SOURCE_MANIFEST_PATH),
    "dataset_sha256": sha256(DATASET_PATH), "source_audit_reference": SOURCE_AUDIT_REFERENCE,
    "code_revision": code_revision, "working_tree_dirty": working_tree_dirty,
    "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in
                 ("cognitive-traffic", "pandas", "numpy", "lightgbm", "scikit-learn", "joblib")},
    "artifact": {"key": f"artifacts/models/{DATASET_ID}/{RUN_ID}/bundle.joblib", "sha256": sha256(artifact_path)},
    "metrics": {"key": f"artifacts/models/{DATASET_ID}/{RUN_ID}/metrics.json", "sha256": sha256(metrics_path)},
    "candidate_mae": float(metrics[metrics.model == "lightgbm_p50"].mae.mean()),
    "baseline_mae": float(metrics[metrics.model == "persistence"].mae.mean()),
    "peak_mae": peak_mae,
    "promotion_status": "requires_independent_checks_and_review",
}
manifest_path = OUTPUT_DIR / "candidate.json"
manifest_path.write_text(json.dumps(candidate, indent=2, allow_nan=False))
print({"candidate": str(manifest_path), "sha256": sha256(manifest_path),
       "serving_alias_changed": False})


## 6. Review and limitations

No output is committed and no training was executed merely by updating this notebook. Before any serving use, independently review dataset rights/mapping, upstream feature availability, target construction, fold leakage, missingness/slices, best-baseline comparison, interval behavior, serving signature and checksum. Current internal walk-forward folds need a purged-label evaluation before claiming leakage-clean model selection. The held-out artifact is not a validated live model simply because serialization succeeded.

Airflow takes `candidate_key`, `evidence_key` and optional measured `champion_mae`. Evidence JSON must contain the exact candidate SHA-256 and `checks` named `leakage`, `data_quality`, `schema_compatibility`, each with actual boolean `passed` and a nonempty `report_reference`. A supplied report reference is an audit input, not cryptographic proof the check ran. Candidate artifacts/metrics are checksummed. Failed/missing checks or benchmark namespace fail closed; accepted review does not change an MLflow alias. Explicit operational promotion remains a separate reviewed action.

A notebook workflow can be syntactically valid without any lawful acquired dataset, successful clean-kernel training, quality improvement, independent review or Vietnamese model champion. Record those gates honestly in the tracker.
